# PPO + NCAP training (Jupyter)

This notebook trains the NCAP circuit (no controller) with PPO on the `foraging` task.
Adjust the parameters in the **Config** cell and run cells sequentially.

In [4]:
# Notebook initialization: ensure `src/` is on `sys.path`, select correct kernel,
# and make `tonic` available before importing project modules.
import sys
from pathlib import Path
# Locate repository root by walking upwards until a 'src' directory is found.
cwd = Path().resolve()
repo = cwd
while repo.parent != repo and not (repo / 'src').is_dir():
    repo = repo.parent
if not (repo / 'src').is_dir():
    raise RuntimeError(f'Could not find repository root containing "src" from {cwd!s}')
src = str(repo / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
print('Inserted src at', src)
print('sys.executable:', sys.executable)
# Check for a .venv python next to the repo (informational only)
venv_python = repo / '.venv' / 'Scripts' / 'python.exe'
print('.venv python exists:', venv_python.exists(), str(venv_python))
# Make tonic available (git clone if missing) and import training helpers.
from macrocircuits.tonic_setup import ensure_tonic
ensure_tonic()
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch device:', device)
if device.type == 'cuda':
    try:
        torch.set_default_device(device)
        print('Set default device to CUDA')
    except AttributeError:
        print('Torch version does not support set_default_device; continuing with device awareness only')
from macrocircuits.training import run_config, train, play_model

Inserted src at C:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\src
sys.executable: c:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\.venv\Scripts\python.exe
.venv python exists: True C:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\.venv\Scripts\python.exe
Torch device: cuda
Set default device to CUDA


In [5]:
# Config: adjust these values as needed
NETWORK = 'ncap'
METHOD = 'ppo'
N_LINKS = 6
TASK = 'foraging'
CONTROLLER = None  # NCAP-only
STEPS = int(5e5)  # larger training like original notebook
PARALLEL = 1

# Build agent/environment/trainer strings via the helper factory
agent, environment, name, trainer = run_config(
    network=NETWORK,
    method=METHOD,
    n_links=N_LINKS,
    task=TASK,
    controller=CONTROLLER,
    steps=STEPS,
    swimmer_kwargs={'use_weight_sharing': True},
)

print('Agent:', agent)
print('Environment:', environment)
print('Experiment name:', name)
print('Trainer:', trainer)

Agent: tonic.torch.agents.PPO(model=ppo_swimmer_model(n_joints=5, critic_sizes=(256, 256), action_noise=0.1, use_weight_sharing=True), actor_updater=tonic.torch.updaters.ClippedRatio(gradient_clip=0.5), critic_updater=tonic.torch.updaters.VRegression(gradient_clip=0.5))
Environment: tonic.environments.ControlSuite("swimmer-foraging", time_feature=True)
Experiment name: ncap_ppo
Trainer: tonic.Trainer(steps=500000, epoch_steps=20000, save_steps=50000)


In [6]:
# Run training (cell will block while training runs)
train(header=None, agent=agent, environment=environment, name=name, trainer=trainer, parallel=PARALLEL)

Config file saved to data\local\experiments\tonic\swimmer-foraging\ncap_ppo\config.yaml


c:\Users\lukav\OneDrive\Documents\CogSci\projects\neuroai-macrocircuits\.venv\Lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


          Time left:  epoch 0:01:12  total 0:37:32          

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [ ]:
# Playback Trained Model
Run this cell after training to render a video from the saved checkpoint.
</VSCode.Cell>
<VSCode.Cell language="python">
# Render the trained model to video from the latest checkpoint
play_model(f'data/local/experiments/tonic/swimmer-foraging/{name}', checkpoint='last')
</VSCode.Cell>
